# Start-to-Finish Cartesian Wave Project

This notebook runs the packaged Cartesian wave-equation generator, inspects the generated project, builds when local tools are available, and reads convergence diagnostics.

[Index](../index.ipynb) | Previous: [Wave Equation and C Code Generation](wave_equation_and_c_codegen.ipynb) | Next: [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)

## Why This Matters

A start-to-finish generated project shows the whole standalone workflow: generate files, build the C code, run the executable, and inspect diagnostics.

## What You Will See

- Generator discovery and generated file inventory.
- A generated-project build and executable run.
- Diagnostic rows from a convergence-factor rerun.

## Table of Contents

1. [Workflow](#Workflow)
2. [Generator discovery](#Generator-discovery)
3. [Temporary project workspace](#Temporary-project-workspace)
4. [Generate and inspect](#Generate-and-inspect)
5. [Build and run](#Build-and-run)
6. [Convergence diagnostics](#Convergence-diagnostics)
7. [Related notebooks](#Related-notebooks)

## Workflow

The generator command is:

```bash
python -m nrpy.examples.wave_equation_cartesian
```

The command creates a generated C project. This notebook runs it in a temporary workspace so existing learner projects are not overwritten.

## Generator Discovery

Discovery uses `importlib.util.find_spec` so the generator is not imported at notebook startup.

In [1]:
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy


print("nrpy import succeeded")

generator_name = "nrpy.examples.wave_equation_cartesian"
generator_spec = importlib.util.find_spec(generator_name)
print("Generator discovered:", generator_spec is not None)

nrpy import succeeded
Generator discovered: True


## Temporary Project Workspace

The temporary workspace object is kept alive across generation, build, run, and diagnostic-inspection cells.

In [2]:
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_wave_cartesian_")
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_cartesian"

print("Temporary workspace created.")
print("Project directory exists before generation:", project_path.exists())

Temporary workspace created.
Project directory exists before generation: False


## Generate and Inspect

The generator produces project files, build rules, generated C functions, and runtime parameter data. The inspection below prints a compact inventory.

In [3]:
generation_completed = False
if generator_spec is None:
    raise RuntimeError(f"Required generator not found: {generator_name}")
else:
    generate = subprocess.run(
        [sys.executable, "-m", generator_name],
        cwd=workspace_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Generator return code:", generate.returncode)
    if generate.stdout.strip():
        print("\n".join(generate.stdout.splitlines()[:16]))
    if generate.returncode != 0:
        if generate.stderr.strip():
            print("Generator stderr excerpt:")
            print("\n".join(generate.stderr.splitlines()[:12]))
        raise RuntimeError("Cartesian wave generator failed.")
    generation_completed = generate.returncode == 0

if project_path.exists():
    project_files = sorted(
        path.relative_to(project_path)
        for path in project_path.rglob("*")
        if path.is_file()
    )
    print("Generated file count:", len(project_files))
    print("First generated files:")
    for relpath in project_files[:18]:
        print(" ", relpath)

    for relpath in project_files:
        if relpath.name in {"Makefile", "main.c", "wave_equation_cartesian.c"}:
            excerpt = (project_path / relpath).read_text(encoding="utf-8", errors="replace").splitlines()[:10]
            print(f"\nExcerpt from {relpath}:")
            print("\n".join(excerpt))
            break
else:
    print("Generated project directory is not present.")

Generator return code: 0
Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
Generated file count: 23
First generated files:
  BHaH_defines.h
  BHaH_function_prototypes.h
  Makefile
  MoL/MoL_free_intermediate_stage_gfs.c
  MoL/MoL_malloc_intermediate_stage_gfs.c
  MoL/MoL_step_forward_in_time.c
  apply_bcs.c
  cmdline_input_and_parfile_parser.c
  commondata_struct_set_to_default.c
  diagnostics.c
  exact_solution_single_Cartesian_point.c
  griddata_free.c
  initial_data.c
  intrinsics/simd_intrinsics.h
  main.c
  numerical_grids_and_timestep.c
  params_struct_set_to_default.c
  progress_indicator.c

Excerpt from Makefile:
CC ?= gcc  # assigns the value CC to gcc only if environment variable CC is not already set

CFLAGS = -std=gnu99 -O2 -march=native -g -Wall -I.
CXXFLAGS = -I. -O2 -g -Wall -Wno-unknown-pragmas -march=native
VALGRIND_CFLAGS = -I. -std=gnu99

## Build and Run

The project build requires `make` and a C compiler. If they are unavailable, the notebook prints the commands a learner can run later.

In [4]:
import shutil


make_tool = shutil.which("make")
compiler_tool = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
can_build = project_path.exists() and make_tool is not None and compiler_tool is not None

print("make available:", make_tool is not None)
print("C compiler available:", compiler_tool is not None)

build_completed = False
build_skipped_for_tools = False
run_completed = False
if not project_path.exists():
    raise RuntimeError("Build cannot proceed because no generated project is available.")
elif not can_build:
    build_skipped_for_tools = True
    print("Build skipped. From the generated project directory, run: make -j2")
    print("Then run: ./wave_equation_cartesian")
    print("For a convergence rerun, run: ./wave_equation_cartesian 2.0")
else:
    build = subprocess.run(
        ["make", "-j2"],
        cwd=project_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Build return code:", build.returncode)
    if build.stdout.strip():
        print("\n".join(build.stdout.splitlines()[:16]))
    if build.returncode != 0:
        if build.stderr.strip():
            print("Build stderr excerpt:")
            print("\n".join(build.stderr.splitlines()[:16]))
        raise RuntimeError("Generated Cartesian wave project build failed.")
    build_completed = build.returncode == 0

    executable = project_path / "wave_equation_cartesian"
    if build_completed and not executable.exists():
        raise RuntimeError("Generated Cartesian wave executable was not created.")
    for args in ([], ["2.0"]):
        label = "default" if not args else "convergence factor 2.0"
        run = subprocess.run(
            [f"./{executable.name}", *args],
            cwd=project_path,
            text=True,
            capture_output=True,
            timeout=180,
        )
        print(f"Run ({label}) return code:", run.returncode)
        if run.stdout.strip():
            print("\n".join(run.stdout.splitlines()[:12]))
        if run.returncode != 0 and run.stderr.strip():
            print("Run stderr excerpt:")
            print("\n".join(run.stderr.splitlines()[:12]))
        if run.returncode != 0:
            raise RuntimeError(f"Generated Cartesian wave run failed: {label}")
        run_completed = True

make available: True
C compiler available: True


Build return code: 0
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc -st

Run (default) return code: 0


Run (convergence factor 2.0) return code: 0


## Convergence Diagnostics

Successful runs write diagnostic text files. The second run, with convergence factor `2.0`, should produce a file whose name records that factor.

In [5]:
if project_path.exists():
    diagnostics = sorted(project_path.glob("out0d-conv_factor*.txt"))
    if diagnostics:
        for diagnostic in diagnostics:
            print("Diagnostic file:", diagnostic.name)
            print("\n".join(diagnostic.read_text(encoding="utf-8", errors="replace").splitlines()[:10]))
    elif run_completed:
        raise RuntimeError("No convergence diagnostic files were found after successful runs.")
    elif build_skipped_for_tools:
        print("No convergence diagnostics are available because build/run work was skipped.")
    else:
        print("No convergence diagnostic files were found.")
else:
    print("No generated project is available for diagnostic inspection.")

Diagnostic file: out0d-conv_factor1.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.991879e+00 3.991879e+00
1.562500e-01 4.061498e-08 2.140944e-05 3.983805e+00 3.983805e+00
4.687500e-01 4.029277e-07 2.042338e-05 3.919867e+00 3.919865e+00
6.250000e-01 7.161221e-07 1.953338e-05 3.864863e+00 3.864860e+00
7.812500e-01 1.107694e-06 1.838455e-05 3.795414e+00 3.795409e+00
9.375000e-01 1.567359e-06 1.698317e-05 3.712441e+00 3.712435e+00
1.250000e+00 2.637266e-06 1.345550e-05 3.510430e+00 3.510420e+00
1.406250e+00 3.214210e-06 1.134935e-05 3.393995e+00 3.393984e+00
1.562500e+00 3.792566e-06 9.030888e-06 3.269195e+00 3.269183e+00
1.875000e+00 4.857928e-06 3.813369e-06 3.000726e+00 3.000711e+00
Diagnostic file: out0d-conv_factor2.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.997967e+00 3.997967e+00
2.343750e-01 6.449302e-09 1.339274e-06 3.979733e+00 3.979733e+00
3.906250e-01 1.815270e-08 1.306709e-06 3.947547e+00 3.947547e+00
6.250000e-01 4.610504e-08 1.226542e-06 3.870305e+00 3.870304e+00
7.81

## Related Notebooks

[Wave equation and C code generation](wave_equation_and_c_codegen.ipynb) explains the symbolic expressions used in the project. [Boundary conditions and convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb) discusses ghost zones, boundary fills, and convergence reruns at the workflow level.

## Where This Leads

- [Wave Equation and C Code Generation](wave_equation_and_c_codegen.ipynb) reviews the prerequisite step.
- [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.